In [8]:
import pandas as pd
import statsmodels.formula.api as smf
import sys
from pathlib import Path
ISLAND_NAME = "tenerife"
ISLAND_CODE = "tfe"

CWD = Path.cwd().resolve()

# If running from islands/<island>, go up two levels to repo root
if CWD.name ==  "notebooks" and CWD.parent.name == "climate_mortality":
    ROOT = CWD.parent
else:
    ROOT = CWD

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("CWD :", CWD)
print("ROOT:", ROOT)
print("src exists?:", (ROOT / "src").exists())

# Carga tu master TFE
df_calima = pd.read_parquet(ROOT/"data/processed"/ISLAND_NAME/f"calima/calima_proxy_weekly_{ISLAND_CODE}_2016_2025.parquet")
print(df_calima.columns.to_list())
df_deaths = pd.read_parquet(ROOT/"data/processed"/ISLAND_NAME/f"deaths/deaths_weekly_{ISLAND_CODE}_2016_2025.parquet")
print(df_deaths.columns.to_list())
df = pd.merge(df_deaths, df_calima, on="week_start", how="inner")
print(df.shape)
print(df.columns.tolist())
# Regresión simple
model = smf.ols("deaths_week ~ calima_proxy_level", data=df).fit()
print(model.summary())

CWD : C:\Users\fdora\RA_Career\Projects\climate_mortality\notebooks
ROOT: C:\Users\fdora\RA_Career\Projects\climate_mortality
src exists?: True
['week_start', 'calima_proxy_score', 'calima_proxy_level']
['week_start', 'deaths_week', 'deaths_missing_week', 'island_code']
(522, 6)
['week_start', 'deaths_week', 'deaths_missing_week', 'island_code', 'calima_proxy_score', 'calima_proxy_level']
                            OLS Regression Results                            
Dep. Variable:            deaths_week   R-squared:                       0.054
Model:                            OLS   Adj. R-squared:                  0.049
Method:                 Least Squares   F-statistic:                     9.869
Date:                Tue, 05 May 2026   Prob (F-statistic):           2.44e-06
Time:                        17:48:43   Log-Likelihood:                -2346.5
No. Observations:                 522   AIC:                             4701.
Df Residuals:                     518   BIC:           

In [9]:
# Regresión simple con variable numerica.
model = smf.ols("deaths_week ~ calima_proxy_score", data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            deaths_week   R-squared:                       0.049
Model:                            OLS   Adj. R-squared:                  0.047
Method:                 Least Squares   F-statistic:                     26.80
Date:                Tue, 05 May 2026   Prob (F-statistic):           3.22e-07
Time:                        18:00:36   Log-Likelihood:                -2347.9
No. Observations:                 522   AIC:                             4700.
Df Residuals:                     520   BIC:                             4708.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept            136.9831      1

R²: 0.049 vs 0.054 — marginalmente peor con score. El modelo categórico captura mejor la no-linealidad (el salto de possible a intense no es proporcional).
β = 17.43 — por cada unidad de calima_proxy_score, +17.4 muertes. Significativo (p < 0.001).
Conclusión práctica: el modelo categórico es ligeramente superior porque la relación calima-mortalidad no es perfectamente lineal. Para la regresión final, usarías calima_proxy_level como categórico.

In [10]:
# Verificar nulls
print(df[["deaths_week", "calima_proxy_level", "calima_proxy_score"]].isnull().sum())

# Eliminar
df_clean = df.dropna(subset=["deaths_week", "calima_proxy_level", "calima_proxy_score"])
print(f"Rows antes: {len(df)} | Rows después: {len(df_clean)}")

# Rerun modelos
model_level = smf.ols("deaths_week ~ calima_proxy_level", data=df_clean).fit()
model_score = smf.ols("deaths_week ~ calima_proxy_score", data=df_clean).fit()

print(model_level.summary())
print(model_score.summary())

deaths_week           0
calima_proxy_level    0
calima_proxy_score    0
dtype: int64
Rows antes: 522 | Rows después: 522
                            OLS Regression Results                            
Dep. Variable:            deaths_week   R-squared:                       0.054
Model:                            OLS   Adj. R-squared:                  0.049
Method:                 Least Squares   F-statistic:                     9.869
Date:                Tue, 05 May 2026   Prob (F-statistic):           2.44e-06
Time:                        18:05:44   Log-Likelihood:                -2346.5
No. Observations:                 522   AIC:                             4701.
Df Residuals:                     518   BIC:                             4718.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025  